# Image classification

Assign a label to a whole image: defect/ok, species, product type, diagnosis.

Two ways to start, from Hello Model: https://sayhellomodel.com/models/image-classification/

**Your data:** the code uses example file names (such as `data.csv`). Upload your own file with the Files panel on the left, or change the name in the code. Run the cells from the top.

## Quick start

No-code / AutoML vision (Teachable Machine, Roboflow, Google Cloud AutoML): Upload labeled images and get a model in minutes, no code required.

In [ ]:
%pip install -q transformers pillow torch

In [ ]:
# Zero-shot with CLIP: no training needed
# pip install transformers pillow torch
from transformers import pipeline
clf = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")
print(clf("leaf.jpg", candidate_labels=["healthy leaf", "leaf with rust disease"]))

## Train your own model

Fine-tuned pretrained CNN/ViT (ConvNeXt, EfficientNet) via timm: Transfer learning gets high accuracy from a few hundred images per class.

In [ ]:
%pip install -q torch torchvision timm

In [ ]:
# pip install torch torchvision timm
import timm, torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

tf = transforms.Compose([transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(),
                         transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
train = datasets.ImageFolder("data/train", tf)
loader = DataLoader(train, batch_size=32, shuffle=True)

model = timm.create_model("convnext_tiny", pretrained=True, num_classes=len(train.classes))
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)

for epoch in range(5):
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        loss = torch.nn.functional.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    print(epoch, loss.item())
torch.save(model.state_dict(), "model.pt")